# LGBM

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split

In [2]:
# File Paths for train and test data
BASE_PATH = r"../playground-series-s5e12/"
TRAIN_PATH = BASE_PATH + "train.csv"
TEST_PATH =  BASE_PATH + "test.csv"

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [24]:
import datetime
import re
import pandas as pd
LOG_FILE = "logger.txt"
def summarize_experiment_logs(log_file=LOG_FILE, sort_by_roc=True, top_n=None):
    """
    Summarize experiments from a logger.txt file.
    
    Args:
        log_file (str): Path to the log file.
        sort_by_roc (bool): Whether to sort by ROC-AUC descending.
        top_n (int or None): If specified, return only the top N experiments.
    
    Returns:
        pd.DataFrame: DataFrame with Timestamp and ROC-AUC Score.
    """
    timestamps = []
    roc_scores = []

    with open(log_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        # Capture timestamp
        if line.startswith("Timestamp:"):
            ts = line.split("Timestamp:")[1].strip()
            timestamps.append(ts)
        # Capture ROC-AUC score
        elif line.startswith("ROC AUC Score:"):
            match = re.findall(r"[\d\.]+", line)
            if match:
                roc_scores.append(float(match[0]))

    # Make DataFrame
    df = pd.DataFrame({
        "Timestamp": timestamps,
        "ROC-AUC Score": roc_scores
    })

    if sort_by_roc:
        df = df.sort_values("ROC-AUC Score", ascending=False).reset_index(drop=True)

    if top_n is not None:
        df = df.head(top_n)

    return df


n_splits = 5

def format_confusion_matrix(cm, labels=None):
    """
    Returns a list of strings representing a nicely formatted confusion matrix.
    """
    if labels is None:
        labels = [str(i) for i in range(cm.shape[0])]
    
    text = []
    # Header
    header = " " * 10 + "".join([f"{lbl:>10}" for lbl in labels])
    text.append(header)
    text.append("-" * (10 + 10*len(labels)))
    
    # Rows
    for i, row in enumerate(cm):
        row_str = f"{labels[i]:<10}" + "".join([f"{int(round(v)):>10}" for v in row])
        text.append(row_str)
    
    return text


def log_experiment_pretty(
    model,
    roc_score,
    avg_report,
    features_used=None,
    conf_mat=None,
    notes=None,
):

    # ---- Params ----
    params = model.get_params()

    # ---- Feature importance ----
    try:
        importances = model.feature_importances_
    except:
        importances = None

    # ---- Build the text block ----
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    text = []
    text.append("=" * 25 + " EXPERIMENT LOG " + "=" * 25)
    text.append(f"Timestamp: {timestamp}\n")
    text.append(f"ROC AUC Score: {roc_score:.5f}\n", )

    # --- Model parameters ---
    text.append("Model Parameters:")
    for k, v in params.items():
        text.append(f"    {k}: {v}")
    text.append("")

    # --- Features used ---
    if features_used:
        text.append(f"Features Used ({len(features_used)}):")
        text.append("    " + ", ".join(features_used))
        text.append("")

    # --- Feature importance ---
    if features_used and importances is not None:
        text.append("Feature Importances:")
        for feat, imp in zip(features_used, importances):
            text.append(f"    {feat}: {float(imp)}")
        text.append("")

    # --- ROC-AUC ---
    text.append(f"ROC-AUC Score: {roc_score}\n")

    # --- Cross-validation report ---
    text.append(f"=== {n_splits}-Fold Cross-Validation Classification Report (Averaged) ===\n")

    # Header
    header = f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'F1-Score':>10}{'Support':>10}"
    text.append(header)
    text.append("-"*50)

    for label, metrics in avg_report.items():
        # Skip non-dict entries like 'accuracy'
        if not isinstance(metrics, dict):
            continue
        
        precision = metrics.get('precision', 0.0)
        recall = metrics.get('recall', 0.0)
        f1 = metrics.get('f1-score', 0.0)
        support = int(metrics.get('support', 0))
        
        line = f"{str(label):<10}{precision:>10.2f}{recall:>10.2f}{f1:>10.2f}{support:>10}"
        text.append(line)

    # Add accuracy separately if exists
    if 'accuracy' in avg_report:
        acc = avg_report['accuracy']
        text.append("-"*50)
        text.append(f"{'Accuracy':<10}{acc:>10.4f}")

    text.append("")


    # Confusion Matric Usage:
    avg_cm_text = format_confusion_matrix(conf_mat)
    text.append("=== Average Confusion Matrix (over CV folds) ===")
    text.extend(avg_cm_text)
    text.append("")  # blank line after

    # --- Additional notes ---
    if notes:
        text.append("Notes:")
        text.append(f"    {notes}\n")

    text.append("-" * 55 + "\n")

    # ---- Append to file ----
    with open(LOG_FILE, "a") as f:
        f.write("\n".join(text))

    print("✔ Logged experiment in readable format to logger.txt")


In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int64  
 11  heart_rate                

In [17]:
for c in CAT_COLS:
    display(train[c].value_counts())

gender
Female    363237
Male      333085
Other       3678
Name: count, dtype: int64

ethnicity
White       386153
Hispanic    129984
Black       106301
Asian        60120
Other        17442
Name: count, dtype: int64

education_level
Highschool      344145
Graduate        261268
Postgraduate     79642
No formal        14945
Name: count, dtype: int64

income_level
Middle          290557
Lower-Middle    178570
Upper-Middle    127836
Low              85803
High             17234
Name: count, dtype: int64

smoking_status
Never      494448
Current    103363
Former     102189
Name: count, dtype: int64

employment_status
Employed      516170
Retired       115735
Unemployed     49787
Student        18308
Name: count, dtype: int64

family_history_diabetes
0    595419
1    104581
Name: count, dtype: int64

hypertension_history
0    572607
1    127393
Name: count, dtype: int64

cardiovascular_history
0    678773
1     21227
Name: count, dtype: int64

In [34]:
NUM_COLS = [
    'age', 
    'alcohol_consumption_per_week', 
    'physical_activity_minutes_per_week', 
    'diet_score', 
    'sleep_hours_per_day', 
    'screen_time_hours_per_day', 
    'bmi', 
    'waist_to_hip_ratio', 
    'systolic_bp', 
    'diastolic_bp', 
    'heart_rate', 
    'cholesterol_total', 
    'hdl_cholesterol', 
    'ldl_cholesterol', 
    'triglycerides', 
]

CAT_COLS = [
    'gender',
    'ethnicity', 
    'education_level', 
    'income_level', 
    'smoking_status', 
    'employment_status',
    'family_history_diabetes', # Binary 1/0
    'hypertension_history', # Binary 1/0
    'cardiovascular_history', # Binary 1/0
    # 'diagnosed_diabetes' # Binary 1/0
]

BIN_COLS = [
    'family_history_diabetes', # Binary 1/0
    'hypertension_history', # Binary 1/0
    'cardiovascular_history', # Binary 1/0
]

TARGET = 'diagnosed_diabetes'


In [7]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import numpy as np


df = train.copy()
df_test = test.copy()

# X = features, y = target
X = df.drop(["id", TARGET], axis=1)
y = df[TARGET]
X_pred_id = df_test["id"]
X_pred = df_test.drop("id", axis=1)

# Identify categorical columns (LGBM supports category dtype)
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Ensure categorical columns are in 'category' dtype for LGBM
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_pred[c] = X_pred[c].astype("category")

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
accuracies = []
f1_scores = []
auc_scores = []

# Store predictions for test data
test_pred_probas = np.zeros(len(X_pred))
predsonly = []

for train_idx, test_idx in skf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # LightGBM Model
    model = LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        objective="binary",
        boosting_type="gbdt",
        class_weight="balanced",    # <-- handles imbalance
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,   
        verbose=-1
    )

    # Train
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="auc",
    )

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Store test predictions
    predsonly.append(model.predict_proba(X_pred))
    test_pred_probas += model.predict_proba(X_pred)[:, 1] / skf.n_splits

    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)

    print(f"ROC AUC Score: {auc:.5f}")
    print(f"Accuracy: {acc:.4f}, F1-score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    fold += 1

print("\n===== Final Cross-Validation Results =====")
print("Mean Accuracy:", np.mean(accuracies))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean Roc Auc Score:", np.mean(auc_scores))



===== Fold 1 =====
ROC AUC Score: 0.72740
Accuracy: 0.6572, F1-score: 0.6990
              precision    recall  f1-score   support

         0.0       0.53      0.69      0.60     52738
         1.0       0.77      0.64      0.70     87262

    accuracy                           0.66    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.66      0.66    140000


===== Fold 2 =====
ROC AUC Score: 0.72592
Accuracy: 0.6573, F1-score: 0.6990
              precision    recall  f1-score   support

         0.0       0.54      0.69      0.60     52738
         1.0       0.77      0.64      0.70     87262

    accuracy                           0.66    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.66      0.66    140000


===== Fold 3 =====
ROC AUC Score: 0.72663
Accuracy: 0.6568, F1-score: 0.6988
              precision    recall  f1-score   support

         0.0       0.53      0.69      0.60     52739
   

### ===== Initial Cross-Validation Results =====
```
Mean Accuracy: 0.6577785714285714
Mean F1 Score: 0.6996131156536629
Mean Roc Auc Score: 0.7271417658271889
Public LB: 0.69743
```

### Trial 2
```
Mean Accuracy: 0.6597842857142856
Mean F1 Score: 0.7036066363362133
Mean Roc Auc Score: 0.727424935864831
Public LB: 0.69590
``

In [26]:
0.69753 - 0.69586, 0.725295063 - 0.7215413

(0.0016699999999999493, 0.0037537629999999655)

In [11]:
predsonly

[array([[0.59890474, 0.40109526],
        [0.46040542, 0.53959458],
        [0.32968356, 0.67031644],
        ...,
        [0.61317009, 0.38682991],
        [0.50626243, 0.49373757],
        [0.5306967 , 0.4693033 ]], shape=(300000, 2)),
 array([[0.59731184, 0.40268816],
        [0.46335552, 0.53664448],
        [0.34887464, 0.65112536],
        ...,
        [0.58785079, 0.41214921],
        [0.50219621, 0.49780379],
        [0.52817633, 0.47182367]], shape=(300000, 2)),
 array([[0.59877824, 0.40122176],
        [0.45109719, 0.54890281],
        [0.34655522, 0.65344478],
        ...,
        [0.62561923, 0.37438077],
        [0.50418769, 0.49581231],
        [0.53405644, 0.46594356]], shape=(300000, 2)),
 array([[0.5948706 , 0.4051294 ],
        [0.4649672 , 0.5350328 ],
        [0.33909126, 0.66090874],
        ...,
        [0.60710632, 0.39289368],
        [0.52443444, 0.47556556],
        [0.53579705, 0.46420295]], shape=(300000, 2)),
 array([[0.59781061, 0.40218939],
        [0.453

In [8]:
pred_arr = np.zeros(300000)

In [9]:
for i in predsonly:
    pred_arr+= i[:,1]/5

In [12]:
test.shape

(300000, 25)

In [10]:
pd.DataFrame({"id":X_pred_id, TARGET:pred_arr}).to_csv("lightgbmsub01.csv", index = False)

In [ ]:
pd.DataFrame({'feature':model.feature_name_, 'importance': model.feature_importances_}).sort_values(by='importance')

,feature,importance
23,cardiovascular_history,1192
22,hypertension_history,2022
21,family_history_diabetes,2033
15,gender,3017
19,smoking_status,9027
20,employment_status,9762
17,education_level,12438
1,alcohol_consumption_per_week,14784
16,ethnicity,20135
18,income_level,22776


In [18]:
# 0.69586 Public Score

In [ ]:
# DART


# model = LGBMClassifier(
#     boosting_type="dart",
#     n_estimators=2500,
#     learning_rate=0.01,
#     num_leaves=128,
#     max_depth=-1,
#     feature_fraction=0.7,
#     bagging_fraction=0.7,
#     bagging_freq=5,
#     min_data_in_leaf=35,
#     lambda_l1=3,
#     lambda_l2=12,
#     class_weight="balanced",
#     objective="binary",
#     max_bin=255,
#     random_state=42,
#     n_jobs=-1,
#     verbose=-1
# )


In [46]:
def bp_risk_score(row):
    sys = row['systolic_bp']
    dia = row['diastolic_bp']

    # Low BP (hypotension)
    if sys < 90 or dia < 60:
        return 3   # high risk
    
    # Normal
    if sys < 120 and dia < 80:
        return 1   # lowest risk
    
    # Elevated
    if 120 <= sys <= 129 and dia < 80:
        return 2
    
    # Stage 1
    if sys >= 130 or dia >= 80:
        if sys < 140 and dia < 90:
            return 3
    
    # Stage 2
    if sys < 180 and dia < 120:
        return 4
    
    # Crisis
    if sys >= 180 or dia >= 120:
        return 5

    return np.nan


def preprocess_pipe(main: pd.DataFrame) -> pd.DataFrame:
    """"PreProcessing steps for classifying diabestes."""
    data = main.copy()
    data['bp_category'] = data.apply(bp_risk_score, axis=1)
    # data = pd.get_dummies(data=data, columns=[x for x in CAT_COLS if x not in BIN_COLS] , drop_first=True)
    return data


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np


df = train.copy()
df_test = test.copy()

df = preprocess_pipe(df)
df_test = preprocess_pipe(df_test)

# X = features, y = target
X = df.drop(["id", TARGET], axis=1)
y = df[TARGET]
X_pred_id = df_test["id"]
X_pred = df_test.drop("id", axis=1)

# Identify categorical columns (LGBM supports category dtype)
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Ensure categorical columns are in 'category' dtype for LGBM
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_pred[c] = X_pred[c].astype("category")

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
accuracies = []
f1_scores = []
auc_scores = []

# Store predictions for test data
test_pred_probas = np.zeros(len(X_pred))
oof_preds_lgb = np.zeros(len(X))
predsonly = []

all_reports = []
avg_cm = np.zeros((2, 2))  # assuming binary classification

for train_idx, test_idx in skf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # LightGBM Model
    model = LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        objective="binary",
        boosting_type="gbdt",
        class_weight="balanced",    # <-- handles imbalance
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,   
        verbose=-1
    )

    # Train
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="auc",
    )

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Store test predictions
    predsonly.append(model.predict_proba(X_pred))
    test_pred_probas += model.predict_proba(X_pred)[:, 1] / skf.n_splits
    oof_preds_lgb[test_idx] = model.predict_proba(X_test)[:, 1]

    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)

    print(f"ROC AUC Score: {auc:.5f}")
    print(f"Accuracy: {acc:.4f}, F1-score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    # Logging
    report_dict = classification_report(y_test, y_pred, output_dict=True)
    all_reports.append(report_dict)
    cm = confusion_matrix(y_test, (y_pred > 0.5).astype(int))
    avg_cm += cm/5


    fold += 1

print(f'====================')
print("OOF AUC: LGBM:", roc_auc_score(y, oof_preds_lgb))
print(f'Overall OOF AUC rounded to 4: {roc_auc_score(y, oof_preds_lgb):.4f}')
# print("OOF AUC: XGB", roc_auc_score(y, oof_preds_xgb))
print(f'====================')

avg_report = {}
for label in all_reports[0].keys():
    if not isinstance(all_reports[0][label], dict):
        continue
    avg_report[label] = {}
    for metric in all_reports[0][label].keys():
        values = [fold[label][metric] for fold in all_reports]
        avg_report[label][metric] = float(np.mean(values))

dev_input = "enhanced bp without ohe"
log_experiment_pretty(
    model=model,
    roc_score= roc_auc_score(y, oof_preds_lgb),
    avg_report=avg_report,
    features_used=X_train.columns.tolist(),
    conf_mat = avg_cm,
    notes=dev_input
)    

print("\n===== Final Cross-Validation Results =====")
print("Mean Accuracy:", np.mean(accuracies))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean Roc Auc Score:", np.mean(auc_scores))



===== Fold 1 =====
ROC AUC Score: 0.72773
Accuracy: 0.6576, F1-score: 0.6995
              precision    recall  f1-score   support

         0.0       0.54      0.69      0.60     52738
         1.0       0.77      0.64      0.70     87262

    accuracy                           0.66    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.66      0.66    140000


===== Fold 2 =====
ROC AUC Score: 0.72575
Accuracy: 0.6570, F1-score: 0.6987
              precision    recall  f1-score   support

         0.0       0.53      0.69      0.60     52738
         1.0       0.77      0.64      0.70     87262

    accuracy                           0.66    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.66      0.66    140000


===== Fold 3 =====
ROC AUC Score: 0.72674
Accuracy: 0.6567, F1-score: 0.6987
              precision    recall  f1-score   support

         0.0       0.53      0.69      0.60     52739
   

In [41]:
# Example usage:
summary_df = summarize_experiment_logs()
print("\n=== Summary of Experiments ===\n")
print(summary_df)


=== Summary of Experiments ===

             Timestamp  ROC-AUC Score
0  2025-12-04 13:19:50        0.72714
1  2025-12-04 13:34:10        0.72714
2  2025-12-04 13:44:20        0.72702
3  2025-12-05 12:49:11        0.72701


In [45]:
import re
import os
nums = []

for f in os.listdir():
    m = re.search(r'^lightgbmsub(\d+)\.csv$', f)
    if m:                     # only process matching files
        nums.append(int(m.group(1)))

next_num = (max(nums) + 1) if nums else 1
next_filename = f"lightgbmsub{next_num:02d}.csv"
pred_arr = np.zeros(300000)
for i in predsonly:
    pred_arr+= i[:,1]/5
pd.DataFrame({"id":X_pred_id, TARGET:pred_arr}).to_csv(next_filename, index = False)